# Eurostat Data Extraction
Content:
- 1 Data Extraction: Approaches
    - 1.1. `eurostat` Python Package | Intro
- 2 Bulk Download
    - 2.1. `eurostat` Library for Bulk Downloads
    - 2.2. 2.2. `requests` Library and Dissemination API Approoach for Bulk Downloads
- 3 Uploading Retrieved Data to PostgreSQL Database
    - 3.1. `psycopg2` Import and Connecting to a Database
    - 3.2. Writing Data from `.csv` or `.tsv` Files to PostgreSQL Database

## 1. Data Extraction: Approaches
There are several ways to extract data from Eurostat.  

Additional detailed instructions on Eurostat API usage is also provided on [the official website](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-detailed-guidelines/asynchronous-api), together with [step by step instructions](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-introduction) and [detailed guide on API Statistics](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-detailed-guidelines/api-statistics).

This notebook provides overview of two approaches to data extraction from Eurostat:
- by using `eurostat` Python library;
- by using API and `requests` Python library.


### 1.1. `eurostat` Python Package | Intro
The library `eurostat` allows to work with the EU data. [Documentation](https://pypi.org/project/eurostat/).

Let's suppose, we want to download data for certain countries and for a certain years:

Libraries to work with Eurostat data:

In [ ]:
import eurostat
import pandas as pd
import io
import os
from dotenv import load_dotenv

`eurostat` allows to fetch data *about* a particular table. Let's assume, we want to work with table `isoc_ci_ifp_iu`("Individuals - Internet Use"). The original table is located [here](https://ec.europa.eu/eurostat/databrowser/product/view/ISOC_CI_IFP_IU).

In [ ]:
# list of the eu countries to work with:
eu_countries = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 
    'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 
    'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

# list of the years for the analysis:
years_list = ['2022', '2023', '2024','2025']

# table to access:
table_eu = 'isoc_ci_ifp_iu'

In [ ]:
# special `eurostat` package methods:

# gets all dimension names (geo, time, indic_is, etc.)
params_list = eurostat.get_pars(table_eu)
print(f"Parameters List: {params_list}")

In [ ]:
# special `eurostat` package methods:

# returns a list of all the values for the table isoc_ci_ifp_iu and
# its dimensions 'geo'
geo_list = eurostat.get_dic(table_eu, 'geo')
print(geo_list) 
# [('AT', 'Austria'), ('BE', 'Belgium')...]
print(type(geo_list))

In [ ]:
# this loop returns all the values for all the parameters for the given table:
for each in params_list:
    print('-------', each, '\n', eurostat.get_dic(table_eu, each), '\n')

In [ ]:
# special `eurostat` package methods:

# returns a list of all indicators existing in the given table:
indic_dic = eurostat.get_dic(table_eu, 'indic_is')
print(indic_dic)

# this shows the codes for different internet activities

In [ ]:
# we can download data for particular geo codes, ind_types, indicators;
# for this we can define a dictonary of parameters:

params = {
    'indic_is': ['I_I3_12'],# 'I_IU3','I_I3_12','I_ILT12','I_IUMT12','I_IUEVR','I_IMT12','I_IUX'], 
    'ind_type': ['IND_TOTAL'],
    'unit': ['PC_IND'],
    'geo': ['DE']
}

# executing special function eurostat.get_data_df will create an API call
# and also will transform the received data into a dataframe:
df_germany = eurostat.get_data_df(table_eu, filter_pars=params)
print('Data Type Is: ', type(df_germany))
print(df_germany.head(10))

# for testing data export to PostgreSQL below:
# df_germany.to_csv('test_germany.csv')

In [ ]:
# the data also can be saved to `.csv` if needed:
# df_germany.to_csv('dee_test.csv', index=False)
#print("CSV file successfully saved!")

**Nota Bene:**  
`eurostat.get_data_df()` function **doesn not** allow to filter for years. It wouldn't take `time` or `TIME_PERIOD` as an argument for filter_param, because 'under the hood' it uses bulk download approach. The problem with years can be solved in the two followinf ways:

In [ ]:
# approach 1: keep the columns needed and drop the rest
metadata_cols = ['freq', 'indic_is', 'unit', 'ind_type', 'geo\\TIME_PERIOD']
target_years = ['2024']  # Add more years here if needed, e.g., ['2023', '2024']

df_filtered = df_germany[metadata_cols + target_years]
df_filtered = df_filtered.rename(columns={'geo\\TIME_PERIOD':'country'})

print(df_filtered)
df_filtered.shape

In [ ]:
# approach 2: reshape/melt the data:
df_long = df_germany.melt(
    id_vars=['freq', 'indic_is', 'unit', 'ind_type', 'geo\\TIME_PERIOD'],
    var_name='year',
    value_name='value'
)
df_long = df_long.rename(columns = {'geo\\TIME_PERIOD':'country'})
df_2024 = df_long[df_long['Year'] == '2024']

print(df_2024)

One can use the same approach to download multiple tables with same/different parameters using a list of tables names + a loop. 

**Important**: In order to prevent the code failing, it might be good to test if the required tables exist. For example:

In [ ]:
try:
    test_df = eurostat.get_data_df('isoc_ci_ifp_iu')
    print("Success! The library is working fine.")
except Exception as e:
    print(f"Failed. It's definitely an API/library version issue: {e}")

## 2. Bulk Download

Since I needed multiple tables and (especially) multiple indicators, bulk download approach would work better for me. The approach described above works perfectly for one or few indicators and for one or few countries. In this project, I decided to download the tables **entirely** and then work on fields, countries, and different indicators on the data cleaning stage in `dbt`.

Bulk download can be done manually: [official website](https://ec.europa.eu/eurostat/data/bulkdownload) provides data updates, search, and also `.tsv` format for all the tables. 

There are **two** approaches to the bulk download from Eurostat website:
- using `eurostat` library mentioned before,
- using `requests` library and connecting to special Eurostat API (Dissemination API Endpoint).

For both of them one would need a list with all the tables and their codes to download.

In [ ]:
# get all the tables names from the file with metadata:
df_tables = pd.read_csv('eurostat_tables.csv')
#print(df_tables.head(2))

# save them as a list
tables_codes = df_tables.iloc[:,0].tolist()
print(tables_codes)

# check if the mentioned tables exist. define a function:

def table_exists(table_code):
    try:
        eurostat.get_pars(table_code)
        print(f'Success! Table {table_code} exists!')
        #return True
    except Exception:
        print(f'We have a problem: Table {table_code} seems not to exist.')
        # return False

In [ ]:
# run this function on all the tables mentioned in the document:

for each_table in tables_codes:
    table_exists(each_table)

### 2.1 `eurostat` Library for Bulk Downloads 

Test on multidimensional table `isoc_ci_ac_i` Internet Activities (58 indicators, 182 units of measure):

In [ ]:
df_activities = eurostat.get_data_df('isoc_ci_ac_i')
print('Data Type Is: ', type(df_activities))
print(df_activities.head(10))
# df_activities.to_csv('activities_test.csv', index=False)

In [ ]:
# size check:
# this one was downloaded using eurostat library:
# df_activities.shape
# >>> (597945, 29)

In [ ]:
# this is the same table but it was manually downloaded:
# df_tsv = pd.read_csv('estat_isoc_ci_ac_i.tsv',sep='\t')
# df_tsv.shape
# >>> (597945, 25) why: some columns are tab + comma separated but the count is correct

### 2.2. `requests` Library and Dissemination API Approoach for Bulk Downloads

This approach doesn't require the usage of `eurostat` library. For this, one needs to import `requests` ([standard Python library for API connections](https://requests.readthedocs.io/en/latest/)). Since the bulk download normally provides 'heavy' files, they come in in `.gz` format, so for that `gzip` [library](https://docs.python.org/3/library/gzip.html) is needed too. 

In [ ]:
import requests
import gzip
import io

In [ ]:
def download_eurostat_bulk(table_code):
    # This is the verified, current Eurostat dissemination API endpoint
    url = f"https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{table_code}/?format=TSV&compressed=true"
    
    print(f"Requesting bulk download from: {url}")
    response = requests.get(url)
    
    if response.status_code == 200:
        try:
            with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
                df = pd.read_csv(f, sep='\t')
                return df
        except Exception as gzip_error:
            print(f"Error unzipping data for {table_code}: {gzip_error}")
            print("Server response snippet:", response.content[:100])
            return None
    else:
        print(f"Eurostat API rejected request. HTTP Status: {response.status_code}")
        return None

Test the function `download_eurostat_bulk` on table `isoc_ci_ifp_fu` Individuals - frequency of internet use:

In [ ]:
# test:
df_freq_int_use = download_eurostat_bulk('isoc_ci_ifp_fu')

if df_freq_int_use is not None:
    print("\nSuccess! The download worked.")
    print(df_freq_int_use.head(3))

Test on table `tin00091` Individuals regularly using the internet. The shape of the table and its structure correspond the original structure and shape.

In [ ]:
df_tin = download_eurostat_bulk('tin00091')
print(df_tin.head(10))
print('Table Size: ', df_tin.shape)

## 3. Uploading Retrieved Data to PostgreSQL Database
So far when downloading data from Eurostat / Eurobarometer websites I was saving the `.csv` or `.tsv` files directly on my hard drive. But this is not an approach for the future data processing: in order to go on with `dbt` data wrangling, I need access the tables on PostgreSQL side. For this I will need one more Python library: `psycopg2`. For more information the [documentation](https://pypi.org/project/psycopg2/) is provided.

In order to use `psycopg2`, the library should be installed in terminal:  
`pip install psycopg2`

### 3.1. `psycopg2` Import and Connecting to a Database
Now, I can import the library and connect to a test database.

In [ ]:
import psycopg2

In [ ]:
# read credentials from .env file:
load_dotenv(dotenv_path='.env')
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")

In [ ]:
# test connection:
conn = psycopg2.connect(f"dbname={db_name} user={db_user}")

In [ ]:
# open a cursor to perform database operations:
cur = conn.cursor()

In [ ]:
# try to create a new table, fill it with some data and fetch it back:
try:
    # create the table
    cur.execute("CREATE TABLE IF NOT EXISTS test (id serial PRIMARY KEY, num integer, data varchar);")
    
    # insert the data
    cur.execute("INSERT INTO test (num, data) VALUES (%s, %s)", (100, "abc'def"))
    
    # select and fetch
    cur.execute("SELECT * FROM test;")
    print(cur.fetchone())
    
    # commit the changes so they actually save to the database!
    conn.commit()

except Exception as e:
    print(f"An error occurred: {e}")
    print("Rolling back the transaction...")
    conn.rollback()


In [ ]:
# important commands: 
# conn.rollback()
cur.close() # closes the cursor
conn.close() # closes the connection to the database

In case one needs to clear the table:

In [ ]:
with psycopg2.connect(f"dbname={db_name} user={db_user}") as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("TRUNCATE TABLE test;")
            conn.commit()
            print('Table cleared!')
        except Exception as e:
            print(f'An error occured: {e}')
            conn.rollback()

Inserting some values and retrieveing data back to test connection:

In [ ]:
with psycopg2.connect(f"dbname={db_name} user={db_user}") as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("INSERT INTO test (num, data) VALUES(%s, %s)", (100, "abc'def"))
            cur.execute("INSERT INTO test (num, data) VALUES (%s, %s)", (200, "ngt'pgh"))
            cur.execute("INSERT INTO test (num, data) VALUES (%s,%s)", (999, "xyz'lol"))
            cur.execute("SELECT * FROM test;")
            conn.commit()
            print('The data is there!')
            cur.execute("SELECT * FROM test;")
            all_rows = cur.fetchall()  # fetchall() brings the data into Python memory safely
            for row in all_rows:
                print(row)
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Rolling back the transaction...")
            conn.rollback()

### 3.2. Writing Data from `.csv` or `.tsv` Files to PostgreSQL Database

Here, as in many situations with using Python, there are two approaches. One is the **slow** one: it inserts the data row by row from a `.csv` file to a PostgreSQL database table. In. my case, it's not the best solution , because there are multiple tables with multiple rows.

The second approach uses special methods `COPY` and `cursor.copy_expert`. Below is a test of this second approach.

In [ ]:
test_csv_path = 'test_germany.csv'
test_csv_path_abs = '/Users/dariaskibo/Desktop/digital_literacy_eu/02_data_retrieval/test_germany.csv'

with psycopg2.connect(f'dbname = {db_name} user={db_user}') as conn:
    with conn.cursor() as cur:
        try:
            # create a table
            cur.execute("""
                CREATE TABLE IF NOT EXISTS csv_test_import (
                    id serial PRIMARY KEY,
                    empty_column integer, 
                    freq varchar,
                    indic_is varchar,
                    unit varchar,
                    ind_type varchar,
                    "geo\\TIME_PERIOD" varchar,
                    "2002" numeric,
                    "2003" numeric,
                    "2004" numeric,
                    "2005" numeric,
                    "2006" numeric,
                    "2007" numeric,
                    "2008" numeric,
                    "2009" numeric,
                    "2010" numeric,
                    "2011" numeric,
                    "2012" numeric,
                    "2013" numeric,
                    "2014" numeric,
                    "2015" numeric,
                    "2016" numeric,
                    "2017" numeric,
                    "2018" numeric,
                    "2019" numeric,
                    "2020" numeric,
                    "2021" numeric,
                    "2022" numeric,
                    "2023" numeric,
                    "2024" numeric,
                    "2025" numeric
                );
            """)
            with open(test_csv_path, 'r', encoding='utf-8') as f:
                sql = """
                    COPY csv_test_import (empty_column, freq, indic_is, unit, ind_type,
                        "geo\\TIME_PERIOD", 
                        "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", 
                        "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", 
                        "2020", "2021", "2022", "2023", "2024", "2025" 
                    )
                    FROM STDIN
                    WITH (FORMAT CSV, HEADER true, DELIMITER ',');
                """
                cur.copy_expert(sql, f)
            conn.commit()
            print(f"Successfully automated bulk import of {test_csv_path}.")
        except Exception as e:
            print(f"Automation failed: {e}.")
            conn.rollback()

Now I need to come up with a solution on how not to pass the columns manually when creating multiple tables. To do it, some functions from `eurostat` library are very helpful.

The set of parameters (columns) for most of the tables is pretty much the same:

In [ ]:
params_list = eurostat.get_pars(table_eu)
print(params_list)

What is missing: columns responsible for years. Lets assume, that all the tables I'm aiming to download, have data for 2002-2025. Here is the full list of all the columns and their datatypes.

In [ ]:
# columns names:
columns_list = ['freq', 'indic_is', 'unit', 'ind_type', 'geo\\TIME_PERIOD']
year_columns = []
for each_year in range(2002, 2026):
    year_columns = year_columns + [str(each_year)]
columns_list = columns_list + year_columns

# columns data types
columns_data_types = ['varchar'] * 5 + ['numeric'] * len(range(2002,2026))

# each column now can be defined:
column_definitions = [f'"{col}"{dtype}' for col, dtype in zip(columns_list, columns_data_types)]

# put them together:
table_structure = ", ".join(column_definitions)


Now we can test this structure by creatin a table with it:

In [ ]:
with psycopg2.connect(f'dbname = {db_name} user={db_user}') as conn:
    with conn.cursor() as cur:
        try:
            cur.execute(f"""CREATE TABLE IF NOT EXISTS test_structure (
                        id serial PRIMARY KEY,
                        {table_structure});""")
            conn.commit()
        except Exception as e:
            print(f"Something didn't work: {e}.")
            conn.rollback()


And now we cat try to use it for creating a new table from a different `.csv` file. We won't have it downloaded to hard drive but retrieved from Eurostat website aka putting the entire process together.

In [ ]:
import io 

with psycopg2.connect(f'dbname = {db_name} user={db_user}') as conn:
    with conn.cursor() as cur:
        try:
            # create a table
            cur.execute(f"""CREATE TABLE IF NOT EXISTS csv_test_import_eu (
                        id serial PRIMARY KEY,
                        {table_structure});""")
            print("Fetching data from Eurostat.")
            df = eurostat.get_data_df('isoc_ci_ifp_iu')
            
            # in-memory text stream
            output_buffer = io.StringIO()
            df.to_csv(output_buffer, index=False, header=True)
            output_buffer.seek(0) # reset the stream pointe to the beginning
            sql = """
                COPY csv_test_import_eu (freq, indic_is, unit, ind_type,
                    "geo\\TIME_PERIOD", 
                    "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", 
                    "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", 
                    "2020", "2021", "2022", "2023", "2024", "2025" 
                )
                FROM STDIN
                WITH (FORMAT CSV, HEADER true, DELIMITER ',');
            """
            print("Importing data into PostgreSQL...")
            cur.copy_expert(sql, output_buffer)
            conn.commit()
            print(f"Successfully automated bulk import from Eurostat.")
        except Exception as e:
            print(f"Automation failed: {e}.")
            conn.rollback()


Unfortunatelly, this chunk of code will work only for the tables with the similar structure: to be precise, with exactly theh same **year** columns. But it's not the case with the Eurostat tables I'm using. Some of them have data way before 2002 (as I hard-coded above), or way after (there are those, starting in 2021 of even later). Every time this code would be ran, it will crash (because there would be no values to put into the table on PostgreSQL).

Another approach is to build **dynamic query** to create tables. See below how it works on a smaller table (`isoc_cisci_ip20`).

In [ ]:
dataset_id = 'isoc_ciegi_pb22'

with psycopg2.connect(f'dbname = {db_name} user = {db_user}') as conn:
    with conn.cursor() as cur:
        try:
            df = eurostat.get_data_df(dataset_id)
            raw_columns = df.columns.tolist()
            output_buffer = io.StringIO()
            df.to_csv(output_buffer,index=False, header=True)
            output_buffer.seek(0)

            structure_elements = []
            for col in raw_columns:
                if str(col).isdigit() and len(str(col)) == 4:
                    structure_elements.append(f'"{col}" numeric')
                else:
                    structure_elements.append(f'"{col}" varchar')
            table_structure = ', '.join(structure_elements)

            cur.execute("DROP TABLE IF EXISTS test_structure")
            cur.execute(f"CREATE TABLE test_structure ({table_structure});")

            formatted_cols = ", ".join([f'"{col}"' for col in raw_columns])
            sql = f"""
                COPY test_structure ({formatted_cols})
                FROM STDIN
                WITH (FORMAT CSV, HEADER true, DELIMITER ',');
            """

            cur.copy_expert(sql, output_buffer)
            conn.commit()
            print(f"Successfully built table and imported {dataset_id} dynamically!")
        except Exception as e:
            print(f"Oooops, something went wrong: {e}")
            conn.rollback()


After developing the script and testing its parts on different tables and for different functionality (e.g., dynamic number of columns), I've developed a Python script `retrieve_eurostat.py` which can be executed in terminal. It will:
- read the table codes from a file `eurostat_tables.csv`;
- make a call to the data source using `eurostat` library;
- upload the retrieved table(s) to PostgreSQL using credentials provided in `.env` file.

The script can be found in the same folder.